# VisionLab: annotate with your Colab credits

This notebook runs a bounded annotation batch on a Colab GPU and downloads a project for local review in VisionLab. It does not start a hosted service or require a Modal account.

**Default: only three images from Marsel204/Dataset.** The sample downloader never clones that repository or downloads annotations.json. Increase the limit only for a dataset ZIP you deliberately upload.

Before opening this notebook, run `python scripts/package_colab.py` in your local VisionLab checkout and download the generated source ZIP. Upload that ZIP in cell 2, so Colab runs the exact implementation from your checkout.

Choose **Runtime → Change runtime type → GPU**. Keep only one notebook processing a given output folder. Completed images are saved individually; saving to Drive allows resume after a session ends. GPU availability and runtime duration depend on your Colab plan and remaining compute units.


In [ ]:
import sys
import torch

if sys.version_info < (3, 12):
    raise RuntimeError("Choose a Colab runtime with Python 3.12 or newer.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime under Runtime → Change runtime type, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


In [ ]:
# Upload visionlab-colab-source.zip created from your local checkout.
import hashlib
import io
import json
import stat
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one VisionLab source ZIP.")
SOURCE_ROOT = Path(tempfile.mkdtemp(prefix="visionlab-source-", dir="/content"))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    members = [m for m in archive.infolist() if not m.is_dir()]
    if len(members) > 1000 or sum(m.file_size for m in members) > 20 * 1024**2:
        raise ValueError("Unexpectedly large source bundle.")
    manifest = json.loads(archive.read("bundle-manifest.json"))
    expected = manifest["files"]
    if manifest.get("version") != 1 or set(m.filename for m in members) != set(expected) | {"bundle-manifest.json"}:
        raise ValueError("Invalid source bundle manifest.")
    if not {"app/services/colab_batch.py", "requirements-colab.txt"} <= set(expected):
        raise ValueError("The source bundle is missing the Colab implementation.")
    for name, digest in expected.items():
        relative = PurePosixPath(name)
        member = archive.getinfo(name)
        target = SOURCE_ROOT.joinpath(*relative.parts)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("Unsafe source bundle entry.")
        if not target.resolve().is_relative_to(SOURCE_ROOT):
            raise ValueError("Source bundle path escapes its directory.")
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError("Source bundle checksum mismatch.")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
sys.path.insert(0, str(SOURCE_ROOT))
print("Loaded source from:", SOURCE_ROOT)
del uploaded


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(SOURCE_ROOT / "requirements-colab.txt")])
# Keep Colab's installed CUDA-enabled torch/torchvision pair.


In [ ]:
# @title Annotation settings
DATA_SOURCE = "GitHub sample (3 images)" # @param ["GitHub sample (3 images)", "Upload image ZIP"]
LIMIT = 3 # @param {type:"integer"}
PIPELINE = "dino-masks" # @param ["dino-masks", "dino-boxes", "yolo-masks", "yolo-boxes", "locate-masks", "locate-boxes"]
CLASSES = "motorcycle, car, bus, truck, minivan, person" # @param {type:"string"}
PROMPTS_JSON = "{}" # @param {type:"string"}
CONFIDENCE = 0.35 # @param {type:"number"}
YOLO_WEIGHTS = "yolo11n.pt" # @param {type:"string"}
SAVE_TO_DRIVE = True # @param {type:"boolean"}
RUN_NAME = "visionlab-three-image-test" # @param {type:"string"}

import re
if LIMIT < 1:
    raise ValueError("LIMIT must be a positive number of images.")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", RUN_NAME):
    raise ValueError("RUN_NAME must contain only letters, numbers, underscores, or hyphens.")
if DATA_SOURCE == "GitHub sample (3 images)" and LIMIT > 3:
    raise ValueError("The example contains only three images. Upload an image ZIP for a larger batch.")

from app.services.colab_batch import make_config
CONFIG = make_config(
    [name.strip() for name in CLASSES.split(",")], pipeline=PIPELINE,
    confidence=CONFIDENCE, prompts=json.loads(PROMPTS_JSON), device="cuda",
    yolo_weights=YOLO_WEIGHTS,
)
print("Maximum images this run:", LIMIT)
print("Pipeline:", PIPELINE)


In [ ]:
from app.services.colab_batch import download_example_sample, extract_image_sample

IMAGE_ROOT = Path(tempfile.mkdtemp(prefix="visionlab-input-", dir="/content"))
if DATA_SOURCE == "GitHub sample (3 images)":
    selected = download_example_sample(IMAGE_ROOT, limit=LIMIT)
else:
    uploaded_images = files.upload()
    if len(uploaded_images) != 1:
        raise ValueError("Upload exactly one ZIP of images.")
    archive_path = IMAGE_ROOT.parent / (IMAGE_ROOT.name + ".zip")
    archive_path.write_bytes(next(iter(uploaded_images.values())))
    selected = extract_image_sample(archive_path, IMAGE_ROOT, limit=LIMIT)
    del uploaded_images
print("Selected images:", len(selected))
for path in selected:
    print(path.relative_to(IMAGE_ROOT))


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/VisionLab") / RUN_NAME
else:
    OUTPUT_ROOT = Path("/content/visionlab-results") / RUN_NAME
OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_ROOT)
print("Rerun with the same classes, pipeline, and run name to resume.")
print("Use a NEW run name when changing the annotation configuration.")


In [ ]:
from app.services.colab_batch import run_batch

RESULT = None
RESULT = run_batch(
    IMAGE_ROOT, OUTPUT_ROOT, CONFIG, limit=LIMIT,
    progress=lambda event: print(json.dumps(event), flush=True),
)
print(json.dumps(RESULT, indent=2))
if RESULT["failed"]:
    print("Some images failed. Inspect the errors above, fix the cause, and rerun to retry only unfinished images.")


In [ ]:
# @title Download results
DOWNLOAD_FORMAT = "visionlab" # @param ["visionlab", "yolo", "coco"]
from app.services.colab_batch import export_results

# This also packages the images that finished before an interrupted run.
ARCHIVES = RESULT["archives"] if RESULT is not None else export_results(OUTPUT_ROOT)
files.download(ARCHIVES[DOWNLOAD_FORMAT])


## Review in VisionLab

For **visionlab**, extract the ZIP on your laptop and select that extracted folder as the dataset in VisionLab. The project contains images, data.yaml, derived labels, and canonical .visionlab annotations. It preserves masks, confidence scores, class IDs, and pending review status.

For **yolo**, extract the ZIP and use its data.yaml. YOLO .txt files contain detection boxes; polygons are retained in the export metadata. **coco** contains annotations.json with polygon segmentation and an images directory. This checkout's COCO importer preserves polygon contours.

Stop the GPU runtime with **Runtime → Disconnect and delete runtime** when finished. No keep-alive mechanism is used. Drive checkpoints survive runtime deletion; files saved only under /content do not. Sessions and GPU access are subject to [Colab's limits](https://research.google.com/colaboratory/faq.html).

The notebook and export pipeline are testable locally. GPU inference must still be validated in your own Colab session; no Colab account or credits are accessed by creating this notebook.
